# T47 - 央行及金融资产负债表数据采集系统

## 项目概述

本系统用于从Wind EDB（经济数据库）自动获取央行及金融机构资产负债表指标。

### 核心功能
1. **增量更新**: 从数据库最大日期的下一天开始获取新数据
2. **自动建表**: 检测表是否存在，不存在则自动创建
3. **动态加列**: 如果新增指标，自动添加新列
4. **UPSERT策略**: 使用 INSERT ... ON DUPLICATE KEY UPDATE 实现插入或更新

### 数据源
- **Wind EDB (Economic Database)**: 万得宏观经济数据库
- **API接口**: WindPy的`w.edb()`方法
- **指标数量**: 约130个指标，分为3组

---

## 1. 环境配置

### 1.1 导入依赖库

In [ ]:
# 标准库
import os
import sys
import json
import logging
from datetime import datetime, timedelta
from typing import Optional, List, Dict, Any, Tuple

# 数据处理
import pandas as pd
import numpy as np

# 数据库
import pymysql
import sqlalchemy
from sqlalchemy import inspect, text, MetaData, Table, Column
from sqlalchemy.pool import NullPool, QueuePool

# Wind API
from WindPy import w

# 配置
from config import (
    DATABASE_URL, TABLE_NAME, TABLE_SCHEMA,
    CENTRAL_BANK_INDICATORS, FINANCIAL_INSTITUTION_INDICATORS, OTHER_FINANCIAL_INDICATORS,
    INDICATOR_GROUPS, DEFAULT_START_DATE,
    LOG_FORMAT, LOG_LEVEL, POOL_SIZE, POOL_RECYCLE
)

print("依赖库导入完成")

### 1.2 配置日志

In [ ]:
# 配置日志
logging.basicConfig(
    level=getattr(logging, LOG_LEVEL),
    format=LOG_FORMAT
)
logger = logging.getLogger('T47_央行及金融资产负债表')

print(f"日志配置完成，级别: {LOG_LEVEL}")

### 1.3 初始化Wind连接

In [ ]:
def init_wind_connection() -> bool:
    """
    初始化Wind API连接
    
    Returns:
        bool: 连接成功返回True，否则返回False
    """
    try:
        error_code = w.start()
        if error_code == 0:
            logger.info("Wind API连接成功")
            print("Wind API连接成功")
            return True
        else:
            logger.error(f"Wind API连接失败，错误代码: {error_code}")
            print(f"Wind API连接失败，错误代码: {error_code}")
            return False
    except Exception as e:
        logger.error(f"Wind API连接异常: {e}")
        print(f"Wind API连接异常: {e}")
        return False

# 初始化Wind连接
wind_connected = init_wind_connection()

### 1.4 创建数据库连接

In [ ]:
def create_db_engine(use_pool: bool = True) -> sqlalchemy.engine.Engine:
    """
    创建数据库连接引擎
    
    Args:
        use_pool: 是否使用连接池
    
    Returns:
        SQLAlchemy Engine对象
    """
    try:
        if use_pool:
            engine = sqlalchemy.create_engine(
                DATABASE_URL,
                poolclass=QueuePool,
                pool_size=POOL_SIZE,
                pool_recycle=POOL_RECYCLE,
                echo=False
            )
        else:
            engine = sqlalchemy.create_engine(
                DATABASE_URL,
                poolclass=NullPool,
                echo=False
            )
        
        # 测试连接
        with engine.connect() as conn:
            conn.execute(text("SELECT 1"))
        
        logger.info("数据库连接成功")
        print("数据库连接成功")
        return engine
        
    except Exception as e:
        logger.error(f"数据库连接失败: {e}")
        print(f"数据库连接失败: {e}")
        raise

# 创建数据库引擎
sql_engine = create_db_engine(use_pool=True)

---

## 2. 数据获取

### 2.1 获取更新日期范围

In [ ]:
def get_update_date_range(engine: sqlalchemy.engine.Engine) -> Tuple[str, str]:
    """
    获取增量更新的日期范围
    
    Args:
        engine: 数据库引擎
    
    Returns:
        Tuple[str, str]: (start_date, end_date) 格式为 'YYYY-MM-DD'
    """
    query = f"""
    SELECT MAX(dt) as dt
    FROM {TABLE_SCHEMA}.{TABLE_NAME}
    """
    
    current_date = datetime.now()
    end_date = current_date.strftime('%Y-%m-%d')
    
    try:
        with engine.begin() as connection:
            result = pd.read_sql(query, con=connection)
        
        max_date = result['dt'].iloc[0]
        
        if pd.isna(max_date):
            # 表为空，使用默认起始日期
            start_date = DEFAULT_START_DATE.strftime('%Y-%m-%d')
            logger.info(f"表为空，从默认日期开始: {start_date}")
        else:
            # 从最大日期+1开始
            start_date = (max_date + timedelta(days=1)).strftime('%Y-%m-%d')
            logger.info(f"增量更新，从 {start_date} 开始")
        
    except Exception as e:
        # 表可能不存在，使用默认起始日期
        start_date = DEFAULT_START_DATE.strftime('%Y-%m-%d')
        logger.warning(f"获取最大日期失败，使用默认日期: {start_date}, 错误: {e}")
    
    print(f"更新日期范围: {start_date} 至 {end_date}")
    return start_date, end_date

# 获取日期范围
dt0, dt1 = get_update_date_range(sql_engine)

### 2.2 Wind API数据获取函数

In [ ]:
def fetch_wind_data(
    indicators: List[str],
    start_date: str,
    end_date: str
) -> Tuple[int, Optional[pd.DataFrame]]:
    """
    从Wind EDB获取数据
    
    Args:
        indicators: 指标代码列表
        start_date: 开始日期 'YYYY-MM-DD'
        end_date: 结束日期 'YYYY-MM-DD'
    
    Returns:
        Tuple[int, DataFrame]: (error_code, data)
            error_code: 0表示成功
            data: DataFrame格式数据，Index为日期，Columns为指标代码
    """
    # 将指标列表转换为逗号分隔的字符串
    codes = ','.join(indicators)
    
    logger.info(f"获取Wind数据: {start_date} 至 {end_date}, 指标数: {len(indicators)}")
    
    try:
        error_code, data = w.edb(codes, start_date, end_date, usedf=True)
        
        if error_code == 0:
            logger.info(f"Wind数据获取成功，数据行数: {len(data) if data is not None else 0}")
            return error_code, data
        else:
            logger.error(f"Wind数据获取失败，错误代码: {error_code}")
            return error_code, None
            
    except Exception as e:
        logger.error(f"Wind API调用异常: {e}")
        return -1, None

print("Wind数据获取函数定义完成")

---

## 3. 数据处理

### 3.1 数据清洗函数

In [ ]:
def clean_data(
    df: pd.DataFrame,
    min_date: str
) -> Optional[pd.DataFrame]:
    """
    数据清洗函数
    
    处理步骤:
    1. 重置索引，将日期转为列
    2. 日期格式转换
    3. 剔除无效日期
    4. 过滤历史数据
    5. 替换NaN为None
    
    Args:
        df: 原始DataFrame，Index为日期
        min_date: 最小日期，早于此日期的数据将被过滤
    
    Returns:
        清洗后的DataFrame，如果无有效数据则返回None
    """
    if df is None or df.empty:
        logger.warning("输入数据为空")
        return None
    
    # Step 1: 重置索引
    df = df.reset_index().rename(columns={'index': 'dt'})
    
    # Step 2: 日期转换
    df['dt'] = pd.to_datetime(df['dt'], errors='coerce')
    
    # Step 3: 剔除无效日期
    df = df.dropna(subset=['dt'])
    
    # Step 4: 过滤历史数据
    df = df[df['dt'] >= pd.to_datetime(min_date)]
    
    # Step 5: 替换NaN为None
    df = df.replace({pd.NA: None, pd.NaT: None})
    df = df.where(pd.notnull(df), None)
    
    if df.empty:
        logger.warning("清洗后数据为空")
        print("No valid data after cleaning.")
        return None
    
    logger.info(f"数据清洗完成，有效行数: {len(df)}")
    return df

print("数据清洗函数定义完成")

### 3.2 测试数据清洗

In [ ]:
# 测试获取第一组数据并清洗
if wind_connected:
    error_code, raw_data = fetch_wind_data(CENTRAL_BANK_INDICATORS[:5], dt0, dt1)
    if error_code == 0 and raw_data is not None:
        print(f"原始数据形状: {raw_data.shape}")
        print(f"原始数据列名: {list(raw_data.columns)[:5]}")
        print("\n原始数据样例:")
        display(raw_data.head())
        
        # 清洗数据
        cleaned_data = clean_data(raw_data, dt0)
        if cleaned_data is not None:
            print(f"\n清洗后数据形状: {cleaned_data.shape}")
            print("\n清洗后数据样例:")
            display(cleaned_data.head())
    else:
        print("获取测试数据失败")
else:
    print("Wind未连接，跳过测试")

---

## 4. 核心逻辑

### 4.1 检查表是否存在

In [ ]:
def check_table_exists(engine: sqlalchemy.engine.Engine) -> bool:
    """
    检查目标表是否存在
    
    Args:
        engine: 数据库引擎
    
    Returns:
        bool: 表存在返回True
    """
    inspector = inspect(engine)
    exists = inspector.has_table(TABLE_NAME)
    
    if exists:
        logger.info(f"表 {TABLE_NAME} 已存在")
    else:
        logger.info(f"表 {TABLE_NAME} 不存在")
    
    return exists

# 检查表是否存在
table_exists = check_table_exists(sql_engine)
print(f"表 '{TABLE_NAME}' 存在: {table_exists}")

### 4.2 自动创建表

In [ ]:
def create_table(engine: sqlalchemy.engine.Engine) -> bool:
    """
    创建目标表
    
    Args:
        engine: 数据库引擎
    
    Returns:
        bool: 创建成功返回True
    """
    create_sql = f"""
    CREATE TABLE {TABLE_SCHEMA}.{TABLE_NAME} (
        dt DATE PRIMARY KEY COMMENT '日期',
        create_time TIMESTAMP DEFAULT CURRENT_TIMESTAMP COMMENT '创建时间',
        update_time TIMESTAMP DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP COMMENT '更新时间'
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='央行及金融资产负债表';
    """
    
    try:
        with engine.begin() as connection:
            connection.execute(text(create_sql))
        
        logger.info(f"表 {TABLE_NAME} 创建成功")
        print(f"表 {TABLE_NAME} 创建成功")
        return True
        
    except Exception as e:
        logger.error(f"表创建失败: {e}")
        print(f"表创建失败: {e}")
        return False

print("创建表函数定义完成")

### 4.3 动态添加列

In [ ]:
def add_missing_columns(
    engine: sqlalchemy.engine.Engine,
    df: pd.DataFrame
) -> int:
    """
    检查并添加缺失的列
    
    Args:
        engine: 数据库引擎
        df: 要插入的DataFrame
    
    Returns:
        int: 新增列数量
    """
    inspector = inspect(engine)
    existing_columns = inspector.get_columns(TABLE_NAME)
    existing_names = {col['name'] for col in existing_columns}
    
    df_columns = set(df.columns)
    new_columns = df_columns - existing_names
    
    added_count = 0
    for col in new_columns:
        if col == 'dt':
            continue
        
        try:
            alter_sql = f"ALTER TABLE {TABLE_SCHEMA}.{TABLE_NAME} ADD COLUMN `{col}` Float COMMENT '{col}';"
            with engine.begin() as connection:
                connection.execute(text(alter_sql))
            
            logger.info(f"新增列: {col}")
            added_count += 1
            
        except Exception as e:
            logger.warning(f"添加列 {col} 失败: {e}")
    
    if added_count > 0:
        print(f"新增 {added_count} 个列")
    
    return added_count

print("动态加列函数定义完成")

### 4.4 UPSERT插入数据

In [ ]:
def upsert_data(
    engine: sqlalchemy.engine.Engine,
    df: pd.DataFrame,
    batch_size: int = 100
) -> Tuple[int, int]:
    """
    使用UPSERT策略插入或更新数据
    
    Args:
        engine: 数据库引擎
        df: 要插入的DataFrame
        batch_size: 批量插入大小
    
    Returns:
        Tuple[int, int]: (成功行数, 失败行数)
    """
    if df is None or df.empty:
        return 0, 0
    
    # 构建列名列表（排除None值的列）
    columns = df.columns.tolist()
    
    # 构建INSERT语句
    insert_columns = ', '.join([f'`{col}`' for col in columns])
    value_placeholders = ', '.join([f':{col}' for col in columns])
    update_columns = ', '.join([f'`{col}` = VALUES(`{col}`)' for col in columns if col != 'dt'])
    
    insert_query = text(f"""
    INSERT INTO {TABLE_SCHEMA}.{TABLE_NAME} ({insert_columns})
    VALUES ({value_placeholders})
    ON DUPLICATE KEY UPDATE {update_columns};
    """)
    
    success_count = 0
    fail_count = 0
    
    # 转换DataFrame为字典列表
    records = df.to_dict('records')
    
    logger.info(f"开始UPSERT {len(records)} 行数据")
    
    with engine.begin() as connection:
        for i, record in enumerate(records):
            try:
                # 处理日期格式
                if 'dt' in record and isinstance(record['dt'], pd.Timestamp):
                    record['dt'] = record['dt'].to_pydatetime().date()
                
                connection.execute(insert_query, record)
                success_count += 1
                
                # 分批提交
                if (i + 1) % batch_size == 0:
                    logger.info(f"已处理 {i + 1}/{len(records)} 行")
                    
            except Exception as e:
                fail_count += 1
                logger.warning(f"第 {i} 行插入失败: {e}")
                
                if fail_count <= 5:
                    print(f"Error inserting row: {record.get('dt', 'unknown')}")
                    print(f"Error: {e}")
    
    logger.info(f"UPSERT完成: 成功 {success_count}, 失败 {fail_count}")
    print(f"UPSERT完成: 成功 {success_count}, 失败 {fail_count}")
    
    return success_count, fail_count

print("UPSERT函数定义完成")

---

## 5. 执行与测试

### 5.1 主处理流程

In [ ]:
def process_indicator_group(
    indicators: List[str],
    group_name: str,
    start_date: str,
    end_date: str,
    engine: sqlalchemy.engine.Engine
) -> Tuple[int, int]:
    """
    处理单个指标组
    
    Args:
        indicators: 指标代码列表
        group_name: 组名（用于日志）
        start_date: 开始日期
        end_date: 结束日期
        engine: 数据库引擎
    
    Returns:
        Tuple[int, int]: (成功行数, 失败行数)
    """
    print(f"\n{'='*50}")
    print(f"处理指标组: {group_name}")
    print(f"指标数量: {len(indicators)}")
    print(f"日期范围: {start_date} 至 {end_date}")
    print(f"{'='*50}")
    
    # Step 1: 获取数据
    error_code, raw_data = fetch_wind_data(indicators, start_date, end_date)
    
    if error_code != 0 or raw_data is None:
        logger.error(f"{group_name}: Wind数据获取失败")
        print(f"{group_name}: Wind数据获取失败")
        return 0, 0
    
    # Step 2: 清洗数据
    cleaned_data = clean_data(raw_data, start_date)
    
    if cleaned_data is None or cleaned_data.empty:
        logger.warning(f"{group_name}: 无有效数据")
        print(f"{group_name}: 无有效数据")
        return 0, 0
    
    # Step 3: 检查表是否存在
    table_exists = check_table_exists(engine)
    
    if not table_exists:
        create_table(engine)
    
    # Step 4: 添加缺失列
    add_missing_columns(engine, cleaned_data)
    
    # Step 5: UPSERT数据
    success, fail = upsert_data(engine, cleaned_data)
    
    print(f"{group_name}: 完成，成功 {success} 行，失败 {fail} 行")
    return success, fail

print("主处理流程函数定义完成")

### 5.2 执行全量更新

In [ ]:
def run_full_update(engine: sqlalchemy.engine.Engine) -> Dict[str, Any]:
    """
    执行全量更新
    
    Args:
        engine: 数据库引擎
    
    Returns:
        Dict: 执行结果统计
    """
    print("\n" + "="*60)
    print("央行及金融资产负债表数据采集系统")
    print("="*60)
    
    start_time = datetime.now()
    results = {}
    total_success = 0
    total_fail = 0
    
    # 获取日期范围
    start_date, end_date = get_update_date_range(engine)
    
    if start_date > end_date:
        print(f"无需更新：起始日期 {start_date} 晚于结束日期 {end_date}")
        return {'status': 'skipped', 'reason': 'no_new_data'}
    
    # 处理每个指标组
    for group_name, indicators in INDICATOR_GROUPS.items():
        success, fail = process_indicator_group(
            indicators=indicators,
            group_name=group_name,
            start_date=start_date,
            end_date=end_date,
            engine=engine
        )
        results[group_name] = {'success': success, 'fail': fail}
        total_success += success
        total_fail += fail
    
    end_time = datetime.now()
    duration = (end_time - start_time).total_seconds()
    
    # 输出汇总
    print("\n" + "="*60)
    print("执行汇总")
    print("="*60)
    for group_name, result in results.items():
        print(f"{group_name}: 成功 {result['success']}, 失败 {result['fail']}")
    print(f"\n总计: 成功 {total_success}, 失败 {total_fail}")
    print(f"耗时: {duration:.2f} 秒")
    print("="*60)
    
    return {
        'status': 'completed',
        'total_success': total_success,
        'total_fail': total_fail,
        'duration': duration,
        'details': results
    }

print("全量更新函数定义完成")

### 5.3 执行更新（需Wind连接）

In [ ]:
# 执行更新
if wind_connected:
    update_result = run_full_update(sql_engine)
    print(f"\n更新结果: {update_result['status']}")
else:
    print("Wind未连接，无法执行更新。请确保Wind终端已启动并登录。")

### 5.4 验证数据

In [ ]:
def verify_data(engine: sqlalchemy.engine.Engine) -> pd.DataFrame:
    """
    验证数据库中的数据
    
    Args:
        engine: 数据库引擎
    
    Returns:
        DataFrame: 数据统计信息
    """
    query = f"""
    SELECT 
        MIN(dt) as min_date,
        MAX(dt) as max_date,
        COUNT(*) as total_rows
    FROM {TABLE_SCHEMA}.{TABLE_NAME}
    """
    
    try:
        with engine.begin() as connection:
            stats = pd.read_sql(query, con=connection)
        
        print("\n数据统计:")
        print(f"  最早日期: {stats['min_date'].iloc[0]}")
        print(f"  最新日期: {stats['max_date'].iloc[0]}")
        print(f"  总行数: {stats['total_rows'].iloc[0]}")
        
        # 获取最近5条数据
        recent_query = f"""
        SELECT * FROM {TABLE_SCHEMA}.{TABLE_NAME}
        ORDER BY dt DESC
        LIMIT 5
        """
        with engine.begin() as connection:
            recent_data = pd.read_sql(recent_query, con=connection)
        
        print("\n最近5条数据:")
        display(recent_data)
        
        return stats
        
    except Exception as e:
        logger.error(f"数据验证失败: {e}")
        print(f"数据验证失败: {e}")
        return None

# 验证数据
verify_data(sql_engine)

---

## 6. 工具函数

### 6.1 获取指标元数据

In [ ]:
def get_indicator_info(indicator_codes: List[str]) -> pd.DataFrame:
    """
    获取Wind指标的元数据信息
    
    Args:
        indicator_codes: 指标代码列表
    
    Returns:
        DataFrame: 指标信息
    """
    codes = ','.join(indicator_codes[:10])  # 限制数量避免超时
    
    try:
        error_code, data = w.edb(codes, "2024-01-01", "2024-01-10", usedf=True)
        if error_code == 0 and data is not None:
            info_df = pd.DataFrame({
                'indicator_code': data.columns,
                'data_points': [data[col].notna().sum() for col in data.columns]
            })
            return info_df
        else:
            return pd.DataFrame()
    except Exception as e:
        print(f"获取指标信息失败: {e}")
        return pd.DataFrame()

print("指标元数据函数定义完成")

### 6.2 数据质量检查

In [ ]:
def check_data_quality(
    engine: sqlalchemy.engine.Engine,
    days: int = 30
) -> Dict[str, Any]:
    """
    检查数据质量
    
    Args:
        engine: 数据库引擎
        days: 检查最近多少天的数据
    
    Returns:
        Dict: 质量检查结果
    """
    query = f"""
    SELECT * FROM {TABLE_SCHEMA}.{TABLE_NAME}
    WHERE dt >= DATE_SUB(CURDATE(), INTERVAL {days} DAY)
    ORDER BY dt
    """
    
    try:
        with engine.begin() as connection:
            df = pd.read_sql(query, con=connection)
        
        if df.empty:
            return {'status': 'no_data', 'message': f'最近{days}天无数据'}
        
        # 计算缺失率
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        missing_rates = df[numeric_cols].isna().mean() * 100
        
        quality_report = {
            'status': 'ok',
            'total_rows': len(df),
            'date_range': (df['dt'].min(), df['dt'].max()),
            'avg_missing_rate': missing_rates.mean(),
            'high_missing_cols': missing_rates[missing_rates > 50].to_dict()
        }
        
        print(f"\n数据质量报告 (最近{days}天):")
        print(f"  总行数: {quality_report['total_rows']}")
        print(f"  日期范围: {quality_report['date_range'][0]} 至 {quality_report['date_range'][1]}")
        print(f"  平均缺失率: {quality_report['avg_missing_rate']:.2f}%")
        
        if quality_report['high_missing_cols']:
            print(f"  高缺失率列 (>50%):")
            for col, rate in quality_report['high_missing_cols'].items():
                print(f"    {col}: {rate:.1f}%")
        
        return quality_report
        
    except Exception as e:
        return {'status': 'error', 'message': str(e)}

# 执行质量检查
quality_result = check_data_quality(sql_engine, days=30)

### 6.3 日期连续性检查

In [ ]:
def check_date_continuity(
    engine: sqlalchemy.engine.Engine
) -> Dict[str, Any]:
    """
    检查日期连续性
    
    Args:
        engine: 数据库引擎
    
    Returns:
        Dict: 连续性检查结果
    """
    query = f"SELECT DISTINCT dt FROM {TABLE_SCHEMA}.{TABLE_NAME} ORDER BY dt"
    
    try:
        with engine.begin() as connection:
            dates_df = pd.read_sql(query, con=connection)
        
        dates = pd.to_datetime(dates_df['dt'])
        
        # 计算日期间隔
        gaps = dates.diff().dropna()
        
        # 检查异常间隔（超过7天）
        large_gaps = gaps[gaps > timedelta(days=7)]
        
        result = {
            'total_dates': len(dates),
            'min_date': dates.min(),
            'max_date': dates.max(),
            'max_gap': gaps.max() if len(gaps) > 0 else None,
            'large_gaps_count': len(large_gaps),
            'is_continuous': len(large_gaps) == 0
        }
        
        print("\n日期连续性检查:")
        print(f"  总日期数: {result['total_dates']}")
        print(f"  日期范围: {result['min_date']} 至 {result['max_date']}")
        print(f"  最大间隔: {result['max_gap']}")
        print(f"  连续性: {'正常' if result['is_continuous'] else f'存在{result["large_gaps_count"]}个较大间隔}'}")
        
        return result
        
    except Exception as e:
        return {'status': 'error', 'message': str(e)}

# 执行连续性检查
continuity_result = check_date_continuity(sql_engine)

---

## 7. 总结

### 7.1 关闭连接

In [ ]:
# 关闭数据库连接
sql_engine.dispose()
print("数据库连接已关闭")

# 关闭Wind连接
try:
    w.stop()
    print("Wind连接已关闭")
except:
    pass

### 7.2 执行报告

本Notebook实现了以下功能：

1. **环境配置**：导入依赖、配置日志、初始化Wind和数据库连接
2. **数据获取**：通过Wind EDB API获取央行及金融资产负债表指标数据
3. **数据处理**：数据清洗、日期转换、NaN处理
4. **核心逻辑**：自动建表、动态加列、UPSERT插入
5. **执行与测试**：主流程执行、结果验证
6. **工具函数**：指标元数据获取、数据质量检查、日期连续性检查

### 关键特性

- **增量更新**：从数据库最大日期+1开始获取，减少API调用
- **自适应表结构**：自动检测并添加新列
- **幂等性**：使用UPSERT策略，多次运行不会产生重复数据
- **数据验证**：提供数据质量和连续性检查功能

### 改进建议

1. 实现批量插入替代逐行插入，提升性能
2. 添加并发调用多组指标
3. 定期进行全量更新以获取历史数据修正
4. 添加邮件通知和监控告警